In [ ]:
import html
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
import pandas as pd
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)
# TODO: write comments for what the code is doing and go over it line by line
# TODO: add the geom parsing and conversion to WKT 
# TODO: turn this into a function 

BASE_WFS = "https://mapserver.traffweb.app/cgi-bin/hounslow/parkmap"
url="https://hounslow.traffweb.app/traffweb/1/TrafficOrders"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
soup = BeautifulSoup(response.text, 'html.parser')
response = requests.get(url,headers=headers,timeout=10)
layer_input = soup.find("input", {"id": "layerconfig"})

if layer_input:
    raw = html.unescape(layer_input.get("value", ""))
    layers = json.loads(raw)
    layer_id_map={
        layer["name"]: layer["defsel"]
        for layer in layers
        if layer["defsel"]
    }
    all_features = []

for layer_name, order_ids in layer_id_map.items():
    
    if layer_name == "signs":
        print(f"  Skipping {layer_name}")
        continue

    print(f"\nFetching {layer_name}...")

    params = {
        "SERVICE": "WFS",
        "VERSION": "1.1.0",
        "REQUEST": "GetFeature",
        "TYPENAME": layer_name,
        "OUTPUTFORMAT": "GML2",
        "SRSNAME": "EPSG:27700",
        "MYORDERS": order_ids,
    }

    wfs_response = requests.get(BASE_WFS, params=params, timeout=120)
    
    if wfs_response.status_code != 200:
        print(f"  Skipping — status {wfs_response.status_code}")
        continue

    try:
        root = ET.fromstring(wfs_response.text)

        ns = {
            "wfs": "http://www.opengis.net/wfs",
            "gml": "http://www.opengis.net/gml",
            "ms":  "http://mapserver.gis.umn.edu/mapserver",
        }

        # Correct path — through gml:featureMember
        features = root.findall(f"gml:featureMember/ms:{layer_name}", ns)
        print(f"  Features found: {len(features)}")

        for feature in features:
            record = {"layer": layer_name, "borough": "hounslow"}
            for child in feature:
                tag = child.tag.split("}")[-1]
                record[tag] = child.text
            all_features.append(record)

    except ET.ParseError as e:
        print(f"  Parse error: {e}")

df = pd.DataFrame(all_features)
print(f"\nTotal features: {len(df)}")
print(df.columns.tolist())
print(df[["layer", "street_name", "restriction", "district"]].head(10))


Fetching ordersr...
  Features found: 446

Fetching ordersl...
  Features found: 24693

Fetching ordersp...
  Features found: 8

Fetching mordersr...
  Features found: 251

Fetching mordersl...
  Features found: 499

Fetching mordersp...
  Features found: 84

Total features: 25981
['layer', 'borough', 'boundedBy', 'msGeometry', 'ogc_fid', 'item_ref', 'order_ref', 'order_type', 'street_name', 'side_of_road', 'locality', 'district', 'ordstart', 'ordfinish', 'ordlocation', 'schedule', 'date_from', 'date_to', 'pre_description', 'times_of_enforcement', 'post_description', 'sp_filename', 'sp_blockname', 'entry_type', 'restriction', 'pm_id', 'order_id', 'order_doc', 'blockimagefile', 'ms_grid', 'type_ref', 'no_of_spaces', 'length', 'tariff_code', 'tariff', 'pbp_code', 'pbp_tariff', 'zone_code', 'order_type_ln', 'side_of_road_ln', 'restriction_ln', 'tariff1', 'tariff2']
     layer      street_name                                   restriction  \
0  ordersr  PLEYDELL AVENUE  Electric Vehicle c

In [54]:
display(df["msGeometry"][(df["restriction"]=='20 minutes no return for 1 hour')])

1302     \n        
25099    \n        
25102    \n        
Name: msGeometry, dtype: str